In [ ]:
%pip install flask-ngrok 
%pip install pyserial
from flask import Flask, jsonify , request
import time
import serial
import requests
import uuid

app = Flask(__name__)
counter = 0
print(counter)
# Serial setup (adjust COM port)
try:
    arduino = serial.Serial('COM11', 9600, timeout=1)
    time.sleep(2)
    print("[INFO] Arduino connected.")
except Exception as e:
    print(f"[ERROR] Could not connect to Arduino: {e}")
    arduino = None
@app.route('/ping', methods=['POST'])  # ✅ POST allowed
def ping():
    
        request_id = uuid.uuid4()
        data = request.get_json()
        print(f"[PC] Received ping from Colab with message: {data} {request_id}")
        # 🧠 Parse simple message logic
        if "lamp" in data['message'].lower() and "on" in data['message'].lower():
            command = "KITCHEN_LAMP_ON"
        elif "lamp" in data.lower() and "off" in data.lower():
            command = "KITCHEN_LAMP_OFF"
        else:
            command = None
       
        # 🔁 Send to Arduino
        if arduino and command:
            arduino.write((command + "\n").encode())
            print(f"[PC] Sent to Arduino: {command}")
            requests.post("http://127.0.0.1:5000/ping", json={"message": command})
            # return jsonify({'status': 'sent', 'cmd': command})
            return
        else:
            return jsonify({'status': 'ignored', 'reason': 'No valid command'})
        
         
    # return jsonify({'status': 'pong', 'echo': data})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)

#ngrok http 5000